<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5509-notebooks/main/_banners/opim5509_banner.svg" width="100%" alt="OPIM 5509 banner"/>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509Files/blob/main/OPIM5509_Module5_Files/notebooks/M5_2b_Word_Embeddings.ipynb)

🔴
<!-- 🎙 DAVE TALKING POINTS — M5 · 7 — Tokenize, pad, and the Embedding layer (count its parameters)
- Say the line first: M5.1 threw word order away. M5.2 KEEPS it - and step one is turning each word into a vector.
- The picture: one-hot = 10,000 columns of mostly zeros, fixed by hand. An embedding = 8 (or 100) dense numbers per word, LEARNED. Similar words end up close together - king - man + woman lands near queen.
- Two ways to get one: learn it along with your task (this video) or borrow a pretrained one (next video).
- Embedding(1000, 64) is just a lookup table: 1,000 words x 64 numbers = 64,000 parameters.
- IMDB comes pre-tokenized in Keras: keep the top 10,000 words, and pad_sequences cuts every review to its first 20 words (short ones get zeros in front).
- Input(20) -> Embedding(10000, 8) = 80,000 -> Flatten 20 x 8 = 160 -> Dense(1) = 161. Total 80,161. The Input layer is the Keras 3 way - input_length is ignored now.
- Validation accuracy peaks at 0.76 (epoch 7) from 20 words per review; best validation loss 0.498 at epoch 5. But Flatten treats each position separately: 'this movie is shit' vs 'this movie is the shit' - recurrent layers fix that in video 9.
-->

# Word embeddings: from one-hot to learned vectors

--------------------------------------------------
**Dr. Dave Wanik - University of Connecticut**

The basic ways to turn words into numbers, from the obvious to the powerful: **one-hot** (one column per word), then **learned embeddings** (a short vector per word, flattened into a Dense network) - and what flattening can't do. The geometry of GloVe is in M5_2a; reading words *in order* is M5_2c.

Adapted from Chollet, *Deep Learning with Python*, Chapter 6, Section 1.

In [2]:
# seed everything: same numbers on my screen and yours
# (and we re-seed right before every model build, so model 2 is as reproducible as model 1)
import keras
import tensorflow as tf

keras.utils.set_random_seed(5509)
tf.config.experimental.enable_op_determinism()

## The big idea
A computer can't read words - it needs numbers. Two ways to turn a word into numbers:
* **one-hot**: a long row of 0s with a single 1 in the word's own column - 10,000 numbers, set by hand, and every word is a stranger to every other word
* **an embedding**: a short row of numbers (8 here) that the model **learns** - words used the same way end up with similar numbers

![word embeddings vs. one hot encoding](https://s3.amazonaws.com/book.keras.io/img/ch6/word_embeddings.png)

## The data: what the model actually sees
Keras ships IMDB movie reviews already turned into numbers - every word is replaced by its rank (1 = most common word). We keep the 10,000 most common words and the last 20 words of each review, then turn two reviews back into words.

In [ ]:
from keras.datasets import imdb
from keras.utils import pad_sequences

max_features = 10000   # keep the 10,000 most common words
maxlen = 20            # cut every review to N = 20 words (pad_sequences keeps the LAST 20)

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)   # each review arrives as a list of word numbers
x_train = pad_sequences(x_train, maxlen=maxlen)    # every review -> exactly 20 numbers
x_test = pad_sequences(x_test, maxlen=maxlen)
print('train:', x_train.shape, '| held-out test:', x_test.shape)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Keras shifts every word's rank by 3: 0 = padding, 1 = start of review, 2 = unknown word
word_index = imdb.get_word_index()
index_word = {0: '<pad>', 1: '<start>', 2: '<unk>'}
for word, rank in word_index.items():
    index_word[rank + 3] = word
label_name = {0: 'negative', 1: 'positive'}

peek_words = []          # we'll reuse these two reviews later (GloVe!)
fig, axes = plt.subplots(2, 1, figsize=(14, 7))
for k in [0, 1]:
    words = []
    for number in x_train[k]:
        words.append(index_word.get(number, '?'))
    peek_words.append(words)
    print('review', k, '| label:', y_train[k], '(' + label_name[y_train[k]] + ')')
    print('   words :', ' '.join(words))
    print('   vector:', x_train[k])
    axes[k].bar(range(1, maxlen + 1), x_train[k])
    axes[k].set_xticks(range(1, maxlen + 1))
    axes[k].set_xticklabels(words, rotation=60)
    axes[k].set_ylabel('word number')
    axes[k].set_title('review ' + str(k) + ' - label ' + str(y_train[k]) + ' (' + label_name[y_train[k]] + ')')
plt.tight_layout()
plt.show()

# THIS is what the model sees: 20 integers. The bar height is just a word's rank - it means nothing yet.
# The Embedding layer is what turns each integer into a useful vector.

## Step 1: one-hot, one word at a time
Give every word its own column out of 10,000 and put a 1 in it. Each of the 20 words becomes a row of 10,000 numbers - all 0, except one spike. Lay the 20 rows end to end and review 0 is **one long vector of 200,000 numbers with just 20 ones in it**.

In [ ]:
# one-hot, word by word: each word is a row of 10,000 numbers - all 0, except a single 1 in that word's column
one_hot = np.zeros((maxlen, max_features))
for slot in range(maxlen):
    one_hot[slot, x_train[0][slot]] = 1
print('one review, one-hot:', one_hot.shape, '=', one_hot.size, 'numbers -', int(one_hot.sum()), 'of them are 1s')
print('all 25,000 training reviews would be', f'{25000 * one_hot.size:,}', 'numbers')

# lay the 20 rows end to end: ONE vector of 200,000 numbers - word 1 is numbers 0-9,999, word 2 is 10,000-19,999, ...
one_vector = one_hot.reshape(-1)
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(range(len(one_vector)), one_vector, color='tab:blue', linewidth=1)
for slot in range(maxlen):
    if slot % 2 == 0:
        ax.axvspan(slot * max_features, (slot + 1) * max_features, color='grey', alpha=0.08)
    spike = slot * max_features + x_train[0][slot]
    ax.text(spike, 1.04, peek_words[0][slot], rotation=60, fontsize=9, ha='left', va='bottom')
ax.set_ylim(-0.05, 1.6)
ax.set_xlim(0, len(one_vector))
ax.set_yticks([0, 1])
ax.set_xlabel('position in the vector (0 - 199,999): each shaded band is one word slot of 10,000 columns')
ax.set_title('review 0, one-hot: ONE vector of 200,000 numbers - 20 spikes, everything else is 0')
plt.tight_layout()
plt.show()

## Step 2: one-hot a whole data set - a strong baseline!
Squash each review's 20 rows into ONE row of 10,000: a 1 in the column of every word that appears anywhere in the review (the bag of words from M5.1 - word order thrown away). To train, we use the **full** reviews, and a plain Dense network on it is hard to beat.

In [ ]:
# multi-hot: one row per review, a 1 in the column of every word it uses (order thrown away)
(full_train, y_full_train), (full_test, y_full_test) = imdb.load_data(num_words=max_features)

def multi_hot(reviews):
    out = np.zeros((len(reviews), max_features), dtype='float32')
    for r in range(len(reviews)):
        for number in reviews[r]:
            out[r, number] = 1.0
    return out

X_hot_train = multi_hot(full_train)
X_hot_test = multi_hot(full_test)
print('one-hot data set:', X_hot_train.shape, '- one row per review, one column per word')

In [ ]:
# what multi-hot does to our same two reviews: squash the 20 rows into ONE row of 10,000
# a 1 in the column of every word used - order and repeats thrown away
fig, axes = plt.subplots(2, 1, figsize=(15, 6))
for k in [0, 1]:
    row = multi_hot([x_train[k]])[0]
    ones = np.nonzero(row)[0]
    print('review', k, '(' + label_name[y_train[k]] + '):', ' '.join(peek_words[k]))
    print('   ', maxlen, 'words, but only', int(row.sum()), 'different ones -> ', int(row.sum()), 'ones out of', max_features, 'columns (repeats share a column)')
    axes[k].plot(range(max_features), row, color='tab:blue', linewidth=1)
    crowded = []
    for col in ones:
        if col < 400:
            crowded.append(index_word.get(col, '?'))     # common words: too close together to label one by one
        else:
            axes[k].text(col, 1.04, index_word.get(col, '?'), rotation=60, fontsize=9, ha='left', va='bottom')
    axes[k].text(450, 1.45, 'packed at the far left: ' + ', '.join(crowded), fontsize=9, va='center')
    axes[k].set_ylim(-0.05, 1.7)
    axes[k].set_xlim(0, max_features)
    axes[k].set_yticks([0, 1])
    axes[k].set_title('review ' + str(k) + ' (' + label_name[y_train[k]] + '), multi-hot: ONE row of 10,000')
axes[1].set_xlabel('vocabulary column (0 - 9,999) - common words sit far left')
plt.tight_layout()
plt.show()

# the model below trains on the FULL reviews (every word, not just the last 20) - same idea, more 1s per row

In [ ]:
from keras.models import Sequential
from keras.layers import Input, Dense
from keras.callbacks import EarlyStopping

keras.utils.set_random_seed(5509)   # re-seed right before the build: same starting weights every run
model = Sequential()
model.add(Input(shape=(max_features,)))
model.add(Dense(16, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(X_hot_train, y_full_train,
                    epochs=50,
                    batch_size=512,
                    validation_split=0.2,
                    callbacks=[early_stop],
                    verbose=0)

# loss curve - always look at this after a fit!
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Loss curve - one-hot (multi-hot) bag of words, full reviews')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()
print('epochs run:', len(loss), '| best validation loss:', round(min(val_loss), 4), 'at epoch', val_loss.index(min(val_loss)) + 1)

# the honest score: HELD-OUT test reviews
from sklearn.metrics import classification_report, f1_score
preds = (model.predict(X_hot_test, verbose=0).ravel() > 0.5).astype(int)
print(classification_report(y_full_test, preds, target_names=['negative', 'positive']))
print('one-hot bag of words (full reviews) - test macro F1:', round(f1_score(y_full_test, preds, average='macro'), 3))

## Step 3: learned embeddings - a short, dense vector per word
One-hot is 10,000 numbers per word and has no idea that *great* and *excellent* mean nearly the same thing. An **Embedding** layer gives every word a short vector (8 numbers) that the model **learns** while it trains. Then `Flatten` lays the 20 word-vectors end to end and a Dense layer makes the call.

In [ ]:
from keras.models import Sequential
from keras.layers import Input, Embedding, Flatten, Dense
from keras.callbacks import EarlyStopping

keras.utils.set_random_seed(5509)   # re-seed right before the build: same starting weights every run
model = Sequential()
model.add(Input(shape=(maxlen,)))           # 20 word numbers in
model.add(Embedding(10000, 8))              # each word -> 8 learned numbers: 20 x 8
model.add(Flatten())                        # 20 x 8 -> one row of 160
model.add(Dense(1, activation='sigmoid'))   # positive or negative
model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# stop when validation loss stops improving, and hand back the BEST epoch's weights
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(x_train, y_train,
                    epochs=50,
                    batch_size=32,
                    validation_split=0.2,
                    callbacks=[early_stop],
                    verbose=0)

# loss curve - always look at this after a fit!
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Loss curve - learned 8-number embeddings, 20 words')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()
print('epochs run:', len(loss), '| best validation loss:', round(min(val_loss), 4), 'at epoch', val_loss.index(min(val_loss)) + 1)

In [ ]:
# the honest score: HELD-OUT test reviews the model never saw while training
from sklearn.metrics import classification_report, f1_score
probs = model.predict(x_test, verbose=0).ravel()
preds = (probs > 0.5).astype(int)
print(classification_report(y_test, preds, target_names=['negative', 'positive']))
print('learned embedding (20 words) - test macro F1:', round(f1_score(y_test, preds, average='macro'), 3))

## Step 4: Flatten locks every word to its slot
The Embedding layer turns each of the 20 words into 8 numbers - a 20 x 8 grid. `Flatten` lays those rows end to end: 160 numbers, and the Dense layer gives each of the 160 its own weight. So it learns *"'excellent' in slot 20"* as a completely separate thing from *"'excellent' in slot 10"* - it never reads the review in order.

In [ ]:
# review 0 after the Embedding layer: 20 words x 8 numbers each
rows = model.layers[0](x_train[0:1]).numpy()[0]
flat = rows.reshape(-1)                       # what Flatten hands the Dense layer: 20 x 8 = 160 numbers
print('embedded review:', rows.shape, '-> flattened:', flat.shape)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].imshow(rows.T, aspect='auto', cmap='coolwarm')
axes[0].set_xticks(range(maxlen))
axes[0].set_xticklabels(peek_words[0], rotation=60)
axes[0].set_ylabel('embedding number (1-8)')
axes[0].set_title('review 0 after the Embedding layer: one column of 8 numbers per word')
axes[1].bar(range(len(flat)), flat)
for slot in range(1, maxlen):
    axes[1].axvline(slot * 8 - 0.5, color='grey', linewidth=0.5)
axes[1].set_title('after Flatten: the same 160 numbers in one long row - word 1 is columns 1-8, word 2 is columns 9-16, ...')
plt.tight_layout()
plt.show()

### What Flatten can't do
Flatten scores every word **alone, in its slot**: *not* in slot 4 can't flip *good* in slot 5. A recurrent layer reads the words **in order** and carries a memory, so *not* changes how *good* is read (M5_2c):

![Flatten vs SimpleRNN](https://raw.githubusercontent.com/drdave-teaching/OPIM5509-notebooks/main/Module5/img/m5_flatten_vs_rnn.gif)

## Where this leaves us
* **One-hot** is easy and makes a strong baseline - but it's huge (10,000 numbers per review) and every word is a stranger to every other word.
* **Learned embeddings** fix that: a short, dense vector per word, and words used the same way end up close together. (For vectors learned from 6 billion words instead of our 25,000 reviews, that's GloVe - M5_2a.)
* But **Flatten locks every word to its slot**, so the model never reads the review *in order*. That's the next notebook: recurrent layers (M5_2c).
* Here IMDB came pre-packaged as numbers. In real projects you start from raw text files - the capstone does exactly that, with real posts from Bluesky.